# Road Following Live (ONNX Model + Stanley + ROS Subscriber + Interactive UI)

This notebook subscribes to the **ROS Camera Topic** (`/csi_cam_0/image_raw`), running **ONNX Model Inference (`road_following_model.onnx`)** and **Stanley Control** on physical JetRacer.
It provides an **Interactive `ipywidgets` UI** to adjust Stanley parameters (gain $k$, throttle, brake gain, steering bias, alpha) in real time while displaying real-time **FPS & Latency**, live camera feed, and recording video to `output_drive.mp4`.

### 1. Setup Environment & Load ONNX Model (`road_following_model.onnx`)

In [ ]:
import os
import sys
import time
from pathlib import Path

# Add parent directory to sys.path
parent_dir = Path.cwd().parent
if str(parent_dir) not in sys.path:
    sys.path.append(str(parent_dir))
if str(Path.cwd()) not in sys.path:
    sys.path.append(str(Path.cwd()))

import onnxruntime as ort

# Locate ONNX model file (.onnx)
model_path = os.path.join(Path.cwd(), "road_following_model.onnx")
if not os.path.exists(model_path):
    model_path = os.path.join(parent_dir, "notebooks", "road_following_model.onnx")

available_providers = ort.get_available_providers()
providers = ['CUDAExecutionProvider'] if 'CUDAExecutionProvider' in available_providers else []
providers.append('CPUExecutionProvider')

if not os.path.exists(model_path):
    print(f"[!] ERROR: ONNX model file '{model_path}' not found!")
    print("👉 Tip: Run 'python export_onnx.py' to generate 'road_following_model.onnx' from your PyTorch model.")
else:
    print(f"[*] Loading ONNX model from: {model_path}")
    try:
        session = ort.InferenceSession(model_path, providers=providers)
    except Exception:
        session = ort.InferenceSession(model_path, providers=['CPUExecutionProvider'])
    
    input_name = session.get_inputs()[0].name
    output_name = session.get_outputs()[0].name
    print(f"[+] Loaded ONNX Session with providers: {session.get_providers()}")


### 2. Initialize ROS Node & JetRacer Hardware (`NvidiaRacecar`)

In [ ]:
import rospy
from sensor_msgs.msg import Image as ROSImage
from jetracer.nvidia_racecar import NvidiaRacecar
try:
    from jetracer.Controller import StanleyController
except ImportError:
    try:
        from Controller import StanleyController
    except ImportError:
        StanleyController = None

# 1. Initialize ROS Node
try:
    rospy.init_node('road_following_live_notebook', anonymous=True, disable_signals=True)
    print("[+] ROS Node initialized successfully!")
except Exception as e:
    print(f"[*] ROS Node notice: {e}")

# 2. Hardware & Controller Setup
car = NvidiaRacecar()
if StanleyController is not None:
    stanley = StanleyController()
    stanley.reset()
else:
    stanley = None
print("[+] JetRacer hardware and Stanley Controller initialized.")


### 3. Interactive Sliders UI & ROS Subscriber Setup

In [ ]:
import cv2
import base64
import ipywidgets
from IPython.display import display
from onnx_runner import JetRacerROSOnnxRunner, bgr8_to_jpeg

# Unregister previous ROS subscriber if active
if 'ros_sub' in globals() and ros_sub is not None:
    try:
        ros_sub.unregister()
    except Exception:
        pass

# Interactive Parameter Sliders
k_slider        = ipywidgets.FloatSlider(min=0.0, max=10.0, step=0.1, value=2.5,  description='Gain (k)', layout=ipywidgets.Layout(width='340px'))
throttle_slider = ipywidgets.FloatSlider(min=0.0, max=1.0,  step=0.01, value=0.20, description='Throttle', layout=ipywidgets.Layout(width='340px'))
brake_slider    = ipywidgets.FloatSlider(min=0.0, max=1.0,  step=0.01, value=0.10, description='Brake Gain', layout=ipywidgets.Layout(width='340px'))
bias_slider     = ipywidgets.FloatSlider(min=-1.0, max=1.0, step=0.01, value=0.0,  description='Steer Bias', layout=ipywidgets.Layout(width='340px'))
alpha_slider    = ipywidgets.FloatSlider(min=0.0, max=1.0,  step=0.05, value=0.4,  description='Alpha', layout=ipywidgets.Layout(width='340px'))

# State Toggle & Display Widgets
state_widget = ipywidgets.ToggleButtons(options=['STOP', 'RUN'], description='Drive State', value='STOP')
camera_html_widget = ipywidgets.HTML(
    value="<p><b>Waiting for ROS Camera Topic...</b></p>",
    layout=ipywidgets.Layout(width='280px', height='280px')
)

video_output_path = os.path.join(Path.cwd(), "output_drive.mp4")

# Frame callback for live HTML UI update with FPS & Latency
def on_live_frame(cv_image, raw_x, raw_y, smoothed_x, steering, dyn_throttle, fps=0.0, latency_ms=0.0):
    h, w = cv_image.shape[:2]
    px = int(w * (smoothed_x / 2.0 + 0.5))
    py = int(h * (raw_y / 2.0 + 0.5)) if raw_y != 0.0 else int(h * 0.5)

    prediction = cv_image.copy()
    cv2.circle(prediction, (px, py), 8, (0, 255, 0), -1)
    cv2.putText(prediction, f"Steer:{steering:+.2f} Thr:{dyn_throttle:.2f}", (10, 25),
                cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2)
    cv2.putText(prediction, f"FPS:{fps:.1f} Latency:{latency_ms:.1f}ms", (10, 50),
                cv2.FONT_HERSHEY_SIMPLEX, 0.45, (0, 255, 255), 1)

    jpeg_bytes = bgr8_to_jpeg(prediction)
    b64 = base64.b64encode(jpeg_bytes).decode('utf-8')
    
    html_str = f'''
    <div style="font-family: monospace; background: #1e1e1e; color: #00ff00; padding: 10px; border-radius: 8px; display: inline-block;">
        <h5 style="margin:0 0 4px 0; color: #ffffff;">JetRacer ONNX Live Stream</h5>
        <p style="margin:2px 0; font-size:11px;"><b>Target X:</b> {raw_x:+.2f} | <b>Smooth X:</b> {smoothed_x:+.2f}</p>
        <p style="margin:2px 0; font-size:11px;"><b>Steering:</b> {steering:+.2f} | <b>Throttle:</b> {dyn_throttle:.2f}</p>
        <p style="margin:2px 0; font-size:11px; color:#00ffff;"><b>FPS:</b> {fps:.1f} | <b>Latency:</b> {latency_ms:.1f}ms</p>
        <img src="data:image/jpeg;base64,{b64}" style="width:224px; height:224px; border:2px solid #00ff00; border-radius:4px; margin-top:4px;" />
    </div>
    '''
    camera_html_widget.value = html_str

# ONNX Runner setup using dynamic slider reading lambdas
runner = JetRacerROSOnnxRunner(
    session=session,
    car=car,
    stanley=stanley,
    k=lambda: k_slider.value,
    throttle=lambda: throttle_slider.value,
    brake_gain=lambda: brake_slider.value,
    bias=lambda: bias_slider.value,
    alpha=lambda: alpha_slider.value,
    video_path=video_output_path,
    video_fps=20.0,
    on_frame=on_live_frame
)

runner.running = False  # Start paused

def on_state_change(change):
    if change['new'] == 'RUN':
        runner.running = True
        if stanley is not None:
            stanley.reset()
        print("[+] ONNX Autonomous Driving ACTIVE!")
    else:
        runner.stop()
        print("[*] ONNX Autonomous Driving STOPPED.")

state_widget.observe(on_state_change, names='value')

topic_name = "/csi_cam_0/image_raw"
ros_sub = rospy.Subscriber(topic_name, ROSImage, runner.image_callback, queue_size=1, buff_size=2**24)

# Assemble Complete Interactive UI Layout
controls_box = ipywidgets.VBox([
    state_widget,
    k_slider,
    throttle_slider,
    brake_slider,
    bias_slider,
    alpha_slider
])

live_ui_widget = ipywidgets.HBox([camera_html_widget, controls_box])
display(live_ui_widget)

print(f"[*] Subscribed to ROS Camera Topic: {topic_name}")
print(f"[*] Video recording configured -> {video_output_path}")


### 4. Emergency Stop Cell

In [ ]:
# Emergency Stop Cell
state_widget.value = 'STOP'
runner.stop()
